# Monitoring, Data Drift & Concept Drift

In [ ]:
# @title Environment setup
!pip install evidently
!git clone https://github.com/nzmonzmp/dataset-ames.git
from evidently import ColumnMapping

from evidently.report import Report
from evidently.metrics.base_metric import generate_column_metrics
from evidently.metric_preset import (
  DataDriftPreset,
  TargetDriftPreset,
  DataQualityPreset,
  RegressionPreset,
)
from evidently.metrics import *

from evidently.test_suite import TestSuite
from evidently.tests.base_test import generate_column_tests
from evidently.test_preset import (
  DataStabilityTestPreset,
  NoTargetPerformanceTestPreset,
  RegressionTestPreset,
)
from evidently.tests import *
import pandas
import numpy
import matplotlib.pyplot as plt
import seaborn
import sklearn.ensemble
import sklearn.model_selection
import scipy.stats
import warnings

warnings.filterwarnings("ignore")


def preprocess(train_file, test_file):
  train_X = pandas.read_csv(train_file, index_col="Id")
  test_X = pandas.read_csv(test_file, index_col="Id")

  train_y = train_X.pop("SalePrice")

  all_X = pandas.concat([train_X, test_X])

  cols_1 = ["LotFrontage"]
  all_X[cols_1] = all_X[cols_1].fillna(train_X[cols_1].median())

  cols_2 = [
    "MSZoning",
    "Electrical",
    "KitchenQual",
    "Exterior1st",
    "Exterior2nd",
    "SaleType",
    "Utilities",
  ]
  all_X[cols_2] = all_X[cols_2].fillna(train_X[cols_2].mode().iloc[0, :])

  cols_4 = [
    "GarageYrBlt",
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtFullBath",
    "BsmtHalfBath",
    "BsmtUnfSF",
    "MasVnrArea",
    "TotalBsmtSF",
  ]
  all_X[cols_4] = all_X[cols_4].fillna(0)

  cols_5 = ["Functional"]
  all_X[cols_5] = all_X[cols_5].fillna("Typ")

  all_X = all_X.fillna("NA")

  cols_numerical2label = ["MSSubClass"]
  all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

  quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
  quality_columns = [
    "BsmtCond",
    "BsmtQual",
    "ExterCond",
    "ExterQual",
    "FireplaceQu",
    "GarageCond",
    "GarageQual",
    "HeatingQC",
    "KitchenQual",
    "PoolQC",
  ]
  street_mapping = dict(NA=0, Grvl=1, Pave=2)
  bsmt_fin_mapping = dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6)

  replace_mapping = dict(
    Alley=street_mapping,
    BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
    BsmtFinType1=bsmt_fin_mapping,
    BsmtFinType2=bsmt_fin_mapping,
    Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
    LandSlope=dict(Sev=1, Mod=2, Gtl=3),
    LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
    PavedDrive=dict(NA=0, N=1, P=2, Y=3),
    Street=dict(Grvl=1, Pave=2),
    Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
  )

  for quality_column in quality_columns:
    replace_mapping[quality_column] = quality_mapping

  all_X.replace(replace_mapping, inplace=True)

  print(f"Number of NAs: {all_X.isnull().sum().sum()}")

  return (
    all_X.iloc[: train_X.shape[0], :],
    train_y,
    all_X.iloc[train_X.shape[0] :, :],
  )

In [ ]:
old, _, new = preprocess("dataset-ames/train.csv", "dataset-ames/test.csv")

## Reports

In [ ]:
report = Report(
  metrics=[
    DataDriftPreset(),
  ]
)

report.run(reference_data=old, current_data=new)
report

## Reports about specific columns

In [ ]:
report = Report(
  metrics=[
    ColumnSummaryMetric(column_name="GrLivArea"),
    ColumnSummaryMetric(column_name="OverallQual"),
  ]
)

report.run(reference_data=old, current_data=new)
report

## Automated tests

In [ ]:
tests = TestSuite(
  tests=[
    NoTargetPerformanceTestPreset(),
  ]
)

tests.run(reference_data=old, current_data=new)
tests

## Automated tests on specific columns

In [ ]:
tests = TestSuite(
  tests=[
    TestColumnDrift("OverallQual"),
    TestColumnDrift("GrLivArea"),
  ]
)

tests.run(reference_data=old, current_data=new)
tests